<a href="https://colab.research.google.com/github/UmerSajid842/Fraud-detection-system/blob/main/Grok_European_credit_card.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommendation for the European Credit Card Fraud dataset
Use a customized hybrid model rather than a pure off-the-shelf model.
The European Credit Card dataset (284,807 transactions, only 492 frauds ≈ 0.172 %, features = Time + Amount + V1–V28 PCA components) has:

Extreme class imbalance
Strong temporal ordering (Time feature)
No explicit card/device/merchant IDs → we must propose a graph structure

A custom architecture that combines Adaptive Transformers (temporal patterns), Graph Proposal Neural Networks (relational structure from time/amount/similarity), and an LLM explanation head directly matches your research title and the data characteristics.
3–4 Recommended Models

Custom Adaptive Spatio-Temporal Graph Transformer + Graph Proposal Network (primary recommendation)Best match to your title.
Temporal Graph Network (TGN) / Temporal Graph AttentionStrong for continuous-time transaction sequences.
GraphSAGE / GAT with time-window graph proposalClassic and effective when graphs are constructed from sliding windows or feature similarity.
FT-Transformer / TabTransformer + Focal Loss (strong tabular baseline)Easy to implement, good performance, but weaker on explicit multi-transaction relational patterns.

Why the custom Adaptive Spatio-Temporal Graph Transformer is mathematically superior

Extreme imbalance: Focal Loss focuses gradient on hard minority examples:$$FL(p_t) = -\alpha_t(1-p_t)^\gamma\log(p_t)$$Combined with graph message passing, rare fraud patterns receive amplified signal.
Temporal dynamics: Time feature allows continuous-time or positional encodings. Adaptive attention weights recent vs. historical transactions:$$\alpha_{ij} = \mathrm{softmax}\Bigl(\frac{Q_iK_j^\top}{\sqrt{d}} + f(\Delta t_{ij})\Bigr)$$
Relational (spatial) structure via Graph Proposal: Even without entity IDs we can propose edges (time proximity, amount similarity, PCA-space nearest neighbors). Graph Attention then aggregates:$$h_v^{(l+1)} = \sigma\Bigl(W\cdot\mathrm{AGG}\{h_u^{(l)}:u\in\mathcal{N}(v)\}\Bigr)$$This captures “fraud bursts” and similar anomalous patterns that pure tabular models miss.
Explainability: Attention weights + local subgraph can be fed to an LLM to produce trustworthy natural-language rationales.

Existing pure models (XGBoost, vanilla Transformer, Autoencoder) are useful baselines but lack the joint spatio-temporal + proposal inductive bias required by your title.


Complete Modular Pipeline for European Credit Card Dataset
Every module’s output is saved as CSV/Excel and becomes the input of the next module. Code is written in separate blocks with detailed comments.

Prerequisites

In [1]:
! pip install pandas numpy scikit-learn torch torch-geometric networkx openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.8 MB/s eta 0:00:00


Module 1: Data Loading & Initial Cleaning
Purpose: Load the classic European credit-card CSV, keep only useful columns, and save a clean version.
Python

In [2]:
# ============================================================
# MODULE 1: DATA LOADING & INITIAL CLEANING
# Purpose: Load creditcard.csv, basic sanity checks, save cleaned raw data.
# ============================================================

import pandas as pd
import numpy as np
import os

os.makedirs("artifacts", exist_ok=True)

def load_european_credit_data(path: str = "creditcard.csv") -> pd.DataFrame:
    """
    Load the standard European Credit Card Fraud dataset.
    Expected columns: Time, V1..V28, Amount, Class
    """
    df = pd.read_csv(path)

    # Basic cleaning
    df = df.drop_duplicates()
    df = df.reset_index(drop=True)

    # Ensure correct dtypes
    df["Class"] = df["Class"].astype(int)

    print(f"Loaded shape: {df.shape}")
    print(f"Fraud rate  : {df['Class'].mean():.6f}")
    print(f"Fraud count : {df['Class'].sum()}")

    return df

# ---------- Execution ----------
# Replace with your actual path
# df_raw = load_european_credit_data("creditcard.csv")

# For demonstration / testing without the full file we create a realistic synthetic sample
np.random.seed(42)
n = 10000
n_fraud = 25
df_raw = pd.DataFrame({
    "Time": np.sort(np.random.randint(0, 172800, n)),          # ~2 days in seconds
    "Amount": np.random.lognormal(3, 1.2, n),
    **{f"V{i}": np.random.randn(n) for i in range(1, 29)},
    "Class": np.zeros(n, dtype=int)
})
df_raw.loc[np.random.choice(n, n_fraud, replace=False), "Class"] = 1

# Save Module 1 output
df_raw.to_csv("artifacts/01_raw_cleaned.csv", index=False)
print("Module 1 complete. Saved → artifacts/01_raw_cleaned.csv")

Module 1 complete. Saved → artifacts/01_raw_cleaned.csv


Module 2: Preprocessing & Feature Engineering
Purpose: Scale Amount & Time, create temporal features, prepare feature matrix and labels.

In [3]:
# ============================================================
# MODULE 2: PREPROCESSING & FEATURE ENGINEERING
# Purpose: Scale numeric features, create time-based features,
#          produce clean X and y, save everything.
# ============================================================

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split

def preprocess_european(df: pd.DataFrame):
    """
    Full preprocessing pipeline tailored to the European Credit Card dataset.
    Returns:
        X, y, feature_names, scaler
    """
    df = df.copy()

    # ----- Temporal features -----
    df["Hour"] = (df["Time"] // 3600) % 24
    df["log_Amount"] = np.log1p(df["Amount"])

    # ----- Feature matrix -----
    feature_cols = ["Time", "Amount", "log_Amount", "Hour"] + [f"V{i}" for i in range(1, 29)]
    # Keep only columns that exist
    feature_cols = [c for c in feature_cols if c in df.columns]

    X = df[feature_cols].copy()
    y = df["Class"].astype(int)

    # Robust scaling is preferred for Amount (outliers are common)
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    X = pd.DataFrame(X_scaled, columns=feature_cols)

    return X, y, feature_cols, scaler

# ---------- Execution ----------
X, y, feature_names, scaler = preprocess_european(df_raw)

# Persist outputs
X.to_csv("artifacts/02_features.csv", index=False)
pd.Series(y, name="Class").to_csv("artifacts/02_labels.csv", index=False)
pd.DataFrame({"feature": feature_names}).to_csv("artifacts/02_feature_names.csv", index=False)

print("Module 2 complete.")
print("Feature matrix shape:", X.shape)
print("Fraud rate:", y.mean())

Module 2 complete.
Feature matrix shape: (10000, 32)
Fraud rate: 0.0025


Module 3: Graph Proposal Network (Key for your title)
Purpose: Because the dataset has no card/device IDs, we propose a graph using time proximity + feature similarity. This is the “Graph Proposal” component.

In [4]:
# ============================================================
# MODULE 3: GRAPH PROPOSAL
# Purpose: Propose edges based on temporal proximity and
#          feature-space similarity. Save edge list + PyG Data object.
# ============================================================

import torch
from torch_geometric.data import Data
from sklearn.neighbors import NearestNeighbors

def propose_graph(X: pd.DataFrame, y: pd.Series, time_col="Time",
                  k_neighbors=8, time_window=3600):
    """
    Graph Proposal strategy:
    1. Connect each transaction to its k nearest neighbors in feature space
       (within a reasonable time window).
    2. This creates a sparse graph that captures “similar anomalous patterns”.
    """
    n = len(X)

    # Use a subset of features for neighbor search (speed + relevance)
    feat_for_nn = X[[c for c in X.columns if c.startswith("V")]].values
    times = X[time_col].values if time_col in X.columns else np.arange(n)

    # Nearest neighbors in PCA space
    nn = NearestNeighbors(n_neighbors=k_neighbors + 1, metric="euclidean", n_jobs=-1)
    nn.fit(feat_for_nn)
    distances, indices = nn.kneighbors(feat_for_nn)

    edge_src, edge_dst = [], []

    for i in range(n):
        for j_idx, j in enumerate(indices[i][1:]):          # skip self
            # Optional: enforce a soft time window
            if abs(times[i] - times[j]) <= time_window * 3:  # allow some flexibility
                edge_src.append(i)
                edge_dst.append(j)
                # undirected
                edge_src.append(j)
                edge_dst.append(i)

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)

    # Node features & labels
    x = torch.tensor(X.values, dtype=torch.float)
    y_tensor = torch.tensor(y.values, dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, y=y_tensor)

    # Save edge list for inspection
    edge_df = pd.DataFrame({"src": edge_src, "dst": edge_dst})
    edge_df.to_csv("artifacts/03_proposed_edges.csv", index=False)
    torch.save(data, "artifacts/03_graph_data.pt")

    print(f"Proposed graph: {data.num_nodes} nodes, {data.num_edges} edges")
    return data

# ---------- Execution ----------
graph_data = propose_graph(X, y, k_neighbors=6, time_window=1800)
print("Module 3 complete. Graph saved.")

Proposed graph: 10000 nodes, 120000 edges
Module 3 complete. Graph saved.


Module 4: Adaptive Spatio-Temporal Model Definition
Purpose: Define the hybrid architecture (Adaptive Transformer + Graph Attention + temporal encoding).

In [5]:
# ============================================================
# MODULE 4: MODEL DEFINITION
# Purpose: Adaptive Spatio-Temporal Graph Transformer
#          - Temporal encoding from Time
#          - Graph Attention (spatial)
#          - Lightweight Transformer encoder
#          - Binary classification head
# ============================================================

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class AdaptiveSpatioTemporalFraudModel(nn.Module):
    """
    Hybrid model matching the research title:
    Adaptive Transformers + Graph Proposal Neural Networks.
    """
    def __init__(self, in_dim: int, hidden_dim: int = 64, n_heads: int = 4, dropout: float = 0.3):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Temporal encoding
        self.time_emb = nn.Linear(1, hidden_dim)

        # Graph Attention layers (spatial)
        self.gat1 = GATConv(hidden_dim, hidden_dim // n_heads, heads=n_heads, dropout=dropout)
        self.gat2 = GATConv(hidden_dim, hidden_dim // n_heads, heads=n_heads, dropout=dropout)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        # Lightweight Transformer encoder (adaptive attention)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_heads,
            dim_feedforward=hidden_dim * 2, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, edge_index, time_feat=None):
        h = self.input_proj(x)

        if time_feat is not None:
            h = h + self.time_emb(time_feat)

        # Spatial message passing with residual
        h = h + F.elu(self.gat1(h, edge_index))
        h = self.norm1(h)
        h = h + F.elu(self.gat2(h, edge_index))
        h = self.norm2(h)

        # Adaptive self-attention over nodes
        h = h.unsqueeze(0)               # [1, N, D]
        h = self.transformer(h)
        h = h.squeeze(0)

        logits = self.classifier(h).squeeze(-1)
        return logits

Module 5: Training Loop (with Focal Loss)
Purpose: Train with Focal Loss + strong positive weighting, save best model and history.

In [6]:
# ============================================================
# MODULE 5: TRAINING
# Purpose: Train the hybrid model with Focal Loss for extreme imbalance.
# ============================================================

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

def train_model(graph_data, X, y, epochs=40, hidden_dim=64):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    idx = np.arange(len(y))
    train_idx, val_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)

    model = AdaptiveSpatioTemporalFraudModel(in_dim=X.shape[1], hidden_dim=hidden_dim).to(device)
    criterion = FocalLoss(alpha=0.75, gamma=2.0)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    x = graph_data.x.to(device)
    edge_index = graph_data.edge_index.to(device)
    y_tensor = graph_data.y.float().to(device)

    # Time feature
    time_feat = torch.tensor(X["Time"].values if "Time" in X.columns else np.zeros(len(X)),
                             dtype=torch.float).unsqueeze(1).to(device)
    time_feat = (time_feat - time_feat.mean()) / (time_feat.std() + 1e-8)

    best_pr = 0.0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, edge_index, time_feat)
        loss = criterion(logits[train_idx], y_tensor[train_idx])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x, edge_index, time_feat)
            val_probs = torch.sigmoid(val_logits[val_idx]).cpu().numpy()
            pr_auc = average_precision_score(y_tensor[val_idx].cpu().numpy(), val_probs)
            roc_auc = roc_auc_score(y_tensor[val_idx].cpu().numpy(), val_probs)

        history.append({"epoch": epoch, "loss": loss.item(), "PR_AUC": pr_auc, "ROC_AUC": roc_auc})

        if pr_auc > best_pr:
            best_pr = pr_auc
            torch.save(model.state_dict(), "artifacts/05_best_model.pt")

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d} | Loss {loss.item():.4f} | Val PR-AUC {pr_auc:.4f} | ROC-AUC {roc_auc:.4f}")

    pd.DataFrame(history).to_csv("artifacts/05_training_history.csv", index=False)
    print(f"Best Val PR-AUC: {best_pr:.4f}")
    return model, train_idx, val_idx, device, time_feat

# ---------- Execution ----------
model, train_idx, val_idx, device, time_feat = train_model(graph_data, X, y, epochs=30)

Device: cpu
Epoch 01 | Loss 0.1756 | Val PR-AUC 0.0024 | ROC-AUC 0.4102
Epoch 05 | Loss 0.1100 | Val PR-AUC 0.0024 | ROC-AUC 0.3974
Epoch 10 | Loss 0.0721 | Val PR-AUC 0.0025 | ROC-AUC 0.4238
Epoch 15 | Loss 0.0543 | Val PR-AUC 0.0024 | ROC-AUC 0.4084
Epoch 20 | Loss 0.0460 | Val PR-AUC 0.0024 | ROC-AUC 0.3933
Epoch 25 | Loss 0.0425 | Val PR-AUC 0.0023 | ROC-AUC 0.3884
Epoch 30 | Loss 0.0419 | Val PR-AUC 0.0023 | ROC-AUC 0.3878
Best Val PR-AUC: 0.0026


Module 6: Complete Fraud Evaluation Code
Purpose: Compute PR-AUC, ROC-AUC, Precision, Recall, F1, MCC at multiple thresholds and save everything.

In [7]:
# ============================================================
# MODULE 6: COMPLETE EVALUATION
# Purpose: Full suite of fraud metrics + predictions saved to CSV.
# ============================================================

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix
)

def evaluate_fraud(y_true, y_prob, threshold=0.3, prefix="val"):
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "PR_AUC"   : average_precision_score(y_true, y_prob),
        "ROC_AUC"  : roc_auc_score(y_true, y_prob),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall"   : recall_score(y_true, y_pred, zero_division=0),
        "F1"       : f1_score(y_true, y_pred, zero_division=0),
        "MCC"      : matthews_corrcoef(y_true, y_pred),
        "Threshold": threshold
    }

    cm = confusion_matrix(y_true, y_pred)
    metrics["TN"], metrics["FP"], metrics["FN"], metrics["TP"] = cm.ravel()

    pd.DataFrame([metrics]).to_csv(f"artifacts/06_{prefix}_metrics_th{threshold}.csv", index=False)
    pd.DataFrame({
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred
    }).to_csv(f"artifacts/06_{prefix}_predictions.csv", index=False)

    print(f"\n===== {prefix.upper()} @ threshold={threshold} =====")
    for k, v in metrics.items():
        print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")
    return metrics

# ---------- Execution ----------
model.load_state_dict(torch.load("artifacts/05_best_model.pt", map_location=device))
model.eval()

x = graph_data.x.to(device)
edge_index = graph_data.edge_index.to(device)

with torch.no_grad():
    logits = model(x, edge_index, time_feat)
    probs = torch.sigmoid(logits).cpu().numpy()

y_np = y.values

print("Probability stats → min / mean / max:", probs.min(), probs.mean(), probs.max())

# Evaluate at several operating points (important for rare fraud)
for th in [0.5, 0.3, 0.2, 0.1]:
    evaluate_fraud(y_np[val_idx], probs[val_idx], threshold=th, prefix="validation")

Probability stats → min / mean / max: 0.36371586 0.41050684 0.5091561

===== VALIDATION @ threshold=0.5 =====
PR_AUC      : 0.0026
ROC_AUC     : 0.4276
Precision   : 0.0000
Recall      : 0.0000
F1          : 0.0000
MCC         : 0.0000
Threshold   : 0.5000
TN          : 1995
FP          : 0
FN          : 5
TP          : 0

===== VALIDATION @ threshold=0.3 =====
PR_AUC      : 0.0026
ROC_AUC     : 0.4276
Precision   : 0.0025
Recall      : 1.0000
F1          : 0.0050
MCC         : 0.0000
Threshold   : 0.3000
TN          : 0
FP          : 1995
FN          : 0
TP          : 5

===== VALIDATION @ threshold=0.2 =====
PR_AUC      : 0.0026
ROC_AUC     : 0.4276
Precision   : 0.0025
Recall      : 1.0000
F1          : 0.0050
MCC         : 0.0000
Threshold   : 0.2000
TN          : 0
FP          : 1995
FN          : 0
TP          : 5

===== VALIDATION @ threshold=0.1 =====
PR_AUC      : 0.0026
ROC_AUC     : 0.4276
Precision   : 0.0025
Recall      : 1.0000
F1          : 0.0050
MCC         : 0.0000
Th

Module 7: LLM-based Trustworthy Explanations
Purpose: Select high-risk transactions and generate human-readable explanations (placeholder ready for a real LLM).

In [8]:
# ============================================================
# MODULE 7: LLM TRUSTWORTHY EXPLANATIONS
# Purpose: Identify high-risk cases and produce explanations.
# ============================================================

def generate_explanation(tid, prob, row):
    """
    Simple transparent explanation.
    Replace the body with a real LLM call (OpenAI / Llama / Grok) later.
    """
    return (
        f"Transaction index {tid} flagged as fraud with probability {prob:.3f}.\n"
        f"Key signals: Amount={row.get('Amount', 'N/A'):.2f}, "
        f"Hour={(row.get('Time', 0)//3600)%24}, "
        f"and anomalous pattern in the PCA feature space (V1–V28).\n"
        f"The Graph Proposal module connected this transaction to similar "
        f"anomalous neighbors, and the Adaptive Transformer confirmed the temporal burst."
    )

# Identify high-risk transactions
risk_threshold = 0.25
high_risk_mask = probs >= risk_threshold
high_risk_indices = np.where(high_risk_mask)[0]
high_risk_probs = probs[high_risk_mask]

# Keep top-K
TOP_K = 10
if len(high_risk_indices) > TOP_K:
    order = np.argsort(high_risk_probs)[::-1][:TOP_K]
    high_risk_indices = high_risk_indices[order]
    high_risk_probs = high_risk_probs[order]

explanations = []
for idx, prob in zip(high_risk_indices, high_risk_probs):
    row = df_raw.iloc[idx]
    exp = generate_explanation(idx, prob, row)
    explanations.append({
        "index": idx,
        "Fraud_Probability": round(float(prob), 4),
        "Explanation": exp
    })

exp_df = pd.DataFrame(explanations)
exp_df.to_csv("artifacts/07_llm_explanations.csv", index=False)

print(f"Saved {len(exp_df)} explanations → artifacts/07_llm_explanations.csv")
print(exp_df.head(3))

Saved 10 explanations → artifacts/07_llm_explanations.csv
   index  Fraud_Probability                                        Explanation
0    911             0.5092  Transaction index 911 flagged as fraud with pr...
1   9863             0.5021  Transaction index 9863 flagged as fraud with p...
2    975             0.4969  Transaction index 975 flagged as fraud with pr...
